# New Training Script – YouTube Title SFT

This notebook contains the code from the `newtrainingscript` folder:
1. **Prep title SFT data** – build data JSONL + train/test ID lists from CSV/JSON
2. **LoRA fine-tuning** – train a HuggingFace model with PEFT/TRL
3. **Generate titles locally** – run inference with base model + optional adapter

## 1. Prep title SFT data (`prep_title_sft_data.py`)

Prepares one data JSONL and train/test ID lists. No separate train/test JSONL; finetune and generate scripts pull from the single data file by ID.

In [ ]:
#!/usr/bin/env python3
"""
Prepare SFT split for YouTube title generation: one data JSONL + train/test ID lists only.
No separate train or test JSONL files; finetune and generate scripts pull from the
single data file by ID.

Inputs:
  - CSV or JSON/JSONL with: video_id (optional), title (required),
    full_transcript/transcript (optional), description (optional)

Outputs:
  - One data JSONL (--out-data): all valid records, one per line:
      {"id": "...", "source": "transcript"|"description", "source_text": "...", "title": "..."}
  - Train IDs (--out-train-ids): one id per line, no overlap with test.
  - Test IDs (--out-test-ids): one id per line.

Prompts are never stored; they are built at training/generation time from this data.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import random
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple


# Quality thresholds (copied from original script)
MIN_TRANSCRIPT_LENGTH = 200
MAX_TRANSCRIPT_LENGTH = 50_000
MIN_DESCRIPTION_LENGTH = 40
MAX_DESCRIPTION_LENGTH = 20_000
MIN_TITLE_LENGTH = 10
MAX_TITLE_LENGTH = 100


def _sha1_int(s: str) -> int:
    return int(hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest(), 16)


def _safe_strip(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    return str(x).strip()


def _load_json_or_jsonl(path: Path) -> List[Dict[str, Any]]:
    if path.suffix.lower() == ".jsonl":
        out: List[Dict[str, Any]] = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                out.append(json.loads(line))
        return out
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data  # type: ignore[return-value]
    raise ValueError(f"Expected list in JSON file: {path}")


def _load_csv(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f)
        return list(reader)


def _infer_id(rec: Dict[str, Any], row_idx: int) -> str:
    vid = _safe_strip(rec.get("video_id"))
    if vid:
        return vid
    url = _safe_strip(rec.get("url"))
    if url:
        return url
    return f"row_{row_idx}"


def _select_source_text(rec: Dict[str, Any]) -> Tuple[str, str]:
    """
    Returns (source, text) where source is 'transcript' or 'description'.
    """
    transcript = _safe_strip(rec.get("full_transcript") or rec.get("transcript"))
    if transcript:
        return "transcript", transcript
    desc = _safe_strip(rec.get("description"))
    if desc:
        return "description", desc
    return "", ""


def _passes_quality(source: str, source_text: str, title: str) -> bool:
    if not title or len(title) < MIN_TITLE_LENGTH or len(title) > MAX_TITLE_LENGTH:
        return False
    if source == "transcript":
        return MIN_TRANSCRIPT_LENGTH <= len(source_text) <= MAX_TRANSCRIPT_LENGTH
    if source == "description":
        return MIN_DESCRIPTION_LENGTH <= len(source_text) <= MAX_DESCRIPTION_LENGTH
    return False


def _to_record(source: str, source_text: str, title: str, rec_id: str) -> Dict[str, Any]:
    """
    Build a minimal, prompt-free training record.

    The training script will later turn this into chat messages and attach
    the system prompt only in memory.
    """
    return {
        "id": rec_id,
        "source": source,
        "source_text": source_text,
        "title": title,
    }


def _prepare_examples(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    examples: List[Dict[str, Any]] = []
    for i, rec in enumerate(records):
        rec_id = _infer_id(rec, i)
        title = _safe_strip(rec.get("title"))
        source, source_text = _select_source_text(rec)
        if not _passes_quality(source, source_text, title):
            continue
        examples.append(_to_record(source, source_text, title, rec_id))
    return examples


def _split_by_id(
    examples: List[Dict[str, Any]],
    test_size: float,
    seed: int,
    stratify_source: bool,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    if not 0.0 < test_size < 1.0:
        raise ValueError("--test-size must be in (0, 1)")

    # Deduplicate by id (keep first; quality filtering already applied)
    by_id: Dict[str, Dict[str, Any]] = {}
    for ex in examples:
        ex_id = str(ex["id"])
        if ex_id not in by_id:
            by_id[ex_id] = ex

    ids = list(by_id.keys())
    rng = random.Random(seed)

    if stratify_source:
        from typing import DefaultDict

        buckets: DefaultDict[str, List[str]] = DefaultDict(list)
        for ex_id, ex in by_id.items():
            buckets[str(ex.get("source", ""))].append(ex_id)

        test_ids: List[str] = []
        for _, b in buckets.items():
            if not b:
                continue
            rng.shuffle(b)
            k = max(1, int(round(len(b) * test_size))) if len(b) >= 2 else 1
            k = min(k, len(b) - 1) if len(b) > 1 else 1
            test_ids.extend(b[:k])
        test_ids = sorted(set(test_ids), key=lambda s: _sha1_int(f"{seed}:{s}"))
        test_set = set(test_ids)
    else:
        rng.shuffle(ids)
        k = int(round(len(ids) * test_size))
        k = max(1, min(k, len(ids) - 1)) if len(ids) > 1 else 1
        test_set = set(ids[:k])

    train, test = [], []
    for ex_id, ex in by_id.items():
        (test if ex_id in test_set else train).append(ex)

    # Stable ordering for reproducibility
    train.sort(key=lambda x: _sha1_int(f"{seed}:train:{x['id']}"))
    test.sort(key=lambda x: _sha1_int(f"{seed}:test:{x['id']}"))
    return train, test


def _write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def main() -> None:
    p = argparse.ArgumentParser(
        description=(
            "Prepare SFT JSONL (without system prompt) and split train/test deterministically."
        )
    )
    p.add_argument(
        "--input",
        default="",
        help=(
            "Path to CSV, JSON, or JSONL with titles + transcripts/descriptions. "
            "If omitted, defaults to data/training_data.json when present; "
            "otherwise data_viral_titles.csv."
        ),
    )
    p.add_argument(
        "--out-data",
        required=True,
        help="Output data JSONL path (single file with all valid records for ID lookup).",
    )
    p.add_argument(
        "--out-train-ids",
        required=True,
        help="Output train ID list path (one id per line).",
    )
    p.add_argument(
        "--out-test-ids",
        required=True,
        help="Output test ID list path (one id per line).",
    )
    p.add_argument(
        "--test-size",
        type=float,
        default=0.1,
        help="Fraction to reserve for test (default: 0.1).",
    )
    p.add_argument(
        "--seed",
        type=int,
        default=1337,
        help="Random seed for deterministic split (default: 1337).",
    )
    p.add_argument(
        "--stratify-source",
        action="store_true",
        help="Stratify split by source (transcript vs description).",
    )
    args = p.parse_args()

    project_root = Path(__file__).resolve().parent.parent
    default_json = project_root / "data" / "training_data.json"
    default_csv = project_root / "data_viral_titles.csv"

    if args.input.strip():
        in_path = Path(args.input)
    else:
        in_path = default_json if default_json.exists() else default_csv
    if not in_path.exists():
        raise SystemExit(f"Input not found: {in_path}")

    if in_path.suffix.lower() == ".csv":
        records = _load_csv(in_path)
    elif in_path.suffix.lower() in {".json", ".jsonl"}:
        records = _load_json_or_jsonl(in_path)
    else:
        raise SystemExit("Unsupported input type. Use .csv, .json, or .jsonl")

    examples = _prepare_examples(records)
    if len(examples) < 20:
        raise SystemExit(f"Too few usable examples after filtering: {len(examples)}")

    train, test = _split_by_id(
        examples=examples,
        test_size=args.test_size,
        seed=args.seed,
        stratify_source=args.stratify_source,
    )

    # Single data JSONL: all valid records (train + test) for lookup by ID.
    all_records = train + test
    out_data_path = Path(args.out_data)
    _write_jsonl(out_data_path, all_records)

    # Train/test ID lists only; no train or test JSONL files.
    train_ids_path = Path(args.out_train_ids)
    test_ids_path = Path(args.out_test_ids)
    train_ids_path.parent.mkdir(parents=True, exist_ok=True)
    test_ids_path.parent.mkdir(parents=True, exist_ok=True)
    with train_ids_path.open("w", encoding="utf-8") as f:
        for ex in train:
            f.write(f"{ex['id']}\n")
    with test_ids_path.open("w", encoding="utf-8") as f:
        for ex in test:
            f.write(f"{ex['id']}\n")

    print(f"Wrote data: {len(all_records)} examples -> {args.out_data}")
    print(f"Wrote train IDs: {len(train)} -> {args.out_train_ids}")
    print(f"Wrote test IDs:  {len(test)} -> {args.out_test_ids}")


# Example: run from notebook (adjust paths as needed)
# import sys
# sys.argv = ['prep_title_sft_data.py', '--out-data', 'data/title_sft.jsonl', '--out-train-ids', 'data/train_ids.txt', '--out-test-ids', 'data/test_ids.txt']
# main()

## 2. LoRA fine-tuning (`finetune_lora.py`)

LoRA fine-tuning for YouTube title generation using HuggingFace Transformers + PEFT + TRL. Reads a single data JSONL and train (and optional eval) ID lists.

In [ ]:
#!/usr/bin/env python3
"""
LoRA fine-tuning for YouTube title generation using HuggingFace Transformers + PEFT + TRL.

Reads a single data JSONL and train (and optional eval) ID lists. No separate
train/test JSONL files; records are pulled from the data file by ID.

Usage:
  --data-jsonl <path>   Single JSONL with all records: id, source, source_text, title.
  --train-ids <path>   One id per line for training.
  --eval-ids <path>     Optional; one id per line for eval loss tracking.
"""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path
from typing import Any, Dict, List


SYSTEM_PROMPT = """You are an expert YouTube title generator. Your task is to create compelling, accurate titles that capture the essence of video content.

Guidelines for generating YouTube titles:
1. Be accurate and truthful - the title must reflect the actual content
2. Make it compelling and click-worthy while staying honest
3. Keep titles concise (ideally 30-70 characters, max 100 characters)
4. Use natural language that viewers would search for
5. Highlight the most interesting or valuable aspect
6. Create a curiosity gap without being misleading

Generate only the title itself, nothing else."""


def _load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def _load_id_list(path: Path) -> List[str]:
    """One id per line, strip whitespace, skip empty."""
    ids: List[str] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            sid = line.strip()
            if sid:
                ids.append(sid)
    return ids


def _data_by_id(rows: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {str(r.get("id", "")): r for r in rows}


def _rows_for_ids(data_by_id: Dict[str, Dict[str, Any]], ids: List[str]) -> List[Dict[str, Any]]:
    """Return rows for given ids in order; skip missing."""
    return [data_by_id[i] for i in ids if i in data_by_id]


def _extract_texts(rows: List[Dict[str, Any]], tokenizer: Any) -> List[str]:
    """
    Turn compact records into chat messages + system prompt, then convert to
    model text using the tokenizer's chat template when available.
    """
    texts: List[str] = []
    for r in rows:
        source = (r.get("source") or "").strip().lower()
        source_text = r.get("source_text") or ""
        title = r.get("title") or ""

        label = "Transcript" if source == "transcript" else "Description"
        user = (
            f"Based on the following video {label.lower()}, generate an accurate and compelling YouTube title.\n\n"
            f"{label}:\n{source_text}\n\n"
            "Generate the YouTube title:"
        )
        msgs_with_system = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": title},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            txt = tokenizer.apply_chat_template(
                msgs_with_system,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            parts = []
            for m in msgs_with_system:
                role = m.get("role", "user")
                content = m.get("content", "")
                parts.append(f"{role.upper()}:\n{content}\n")
            txt = "\n".join(parts)
        texts.append(txt)
    return texts


def _infer_target_modules(model: Any) -> List[str]:
    """
    Best-effort inference of common attention projection module names for LoRA.
    Users can override via --target-modules.
    """
    names = set()
    for n, _ in model.named_modules():
        if n.endswith(("q_proj", "k_proj", "v_proj", "o_proj")):
            names.add(n.split(".")[-1])
        if n.endswith(("c_attn", "c_proj")):  # GPT-2 style
            names.add(n.split(".")[-1])
        if n.endswith(("Wqkv", "Wo")):  # some architectures
            names.add(n.split(".")[-1])
    out = sorted(names)
    return out or ["q_proj", "v_proj"]


def _make_sft_collator_with_token_type_ids(tokenizer: Any, max_seq_length: int) -> Any:
    """
    Data collator for SFT that adds token_type_ids (e.g. for Gemma 3), which
    some models require when training. Uses right-padding and fixed length so
    shift_logits / shift_attention_mask shapes stay consistent (avoids IndexError
    when effective sequence length is 0 or 1).
    """
    import torch  # type: ignore[import-untyped]

    pad_token_id = getattr(tokenizer, "pad_token_id", None) or getattr(
        tokenizer, "eos_token_id", 0
    )

    def _collate(features: List[Dict[str, Any]]) -> Dict[str, Any]:
        texts = [f.get("text", "") or "" for f in features]
        # Skip empty texts so we never pass zero-length sequences.
        texts = [t if t.strip() else " " for t in texts]
        # Gemma 3 and many models expect right-padding for training; avoid left padding.
        old_padding_side = getattr(tokenizer, "padding_side", "right")
        tokenizer.padding_side = "right"
        try:
            batch = tokenizer(
                texts,
                padding="max_length",
                truncation=True,
                max_length=max_seq_length,
                return_tensors="pt",
                return_attention_mask=True,
            )
        finally:
            tokenizer.padding_side = old_padding_side
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask", input_ids.ne(pad_token_id).long())
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        batch["labels"] = labels
        # Required by Gemma 3 (and some other models) when training.
        batch["token_type_ids"] = torch.zeros_like(input_ids, dtype=torch.long)
        return batch

    return _collate


def main() -> None:
    p = argparse.ArgumentParser(
        description=(
            "LoRA fine-tune a HF base model for title generation "
            "with the system prompt injected at training time."
        )
    )
    p.add_argument(
        "--data-jsonl",
        required=True,
        help="Data JSONL with all records (id, source, source_text, title).",
    )
    p.add_argument(
        "--train-ids",
        required=True,
        help="Train ID list (one id per line).",
    )
    p.add_argument(
        "--eval-ids",
        default="",
        help="Optional eval ID list for loss tracking (one id per line).",
    )
    p.add_argument(
        "--base-model",
        required=True,
        help="HF model name or local path.",
    )
    p.add_argument(
        "--run-name",
        required=True,
        help="Run name; outputs saved under --output-root/<run-name>/",
    )
    p.add_argument(
        "--output-root",
        default="runs",
        help="Root directory for runs (default: runs).",
    )

    # LoRA
    p.add_argument("--lora-r", type=int, default=8)
    p.add_argument("--lora-alpha", type=int, default=32)
    p.add_argument("--lora-dropout", type=float, default=0.1)
    p.add_argument(
        "--target-modules",
        default="",
        help="Comma-separated target module names (override inference).",
    )

    # Training hyperparams
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument(
        "--batch-size",
        type=int,
        default=1,
        help="Per-device batch size (default: 1).",
    )
    p.add_argument(
        "--grad-accum",
        type=int,
        default=16,
        help="Gradient accumulation steps (default: 16).",
    )
    p.add_argument("--max-seq-len", type=int, default=1024)
    p.add_argument("--seed", type=int, default=1337)

    # Precision / memory
    p.add_argument("--bf16", action="store_true", help="Use bf16 if available.")
    p.add_argument("--fp16", action="store_true", help="Use fp16 if available.")
    p.add_argument(
        "--gradient-checkpointing",
        action="store_true",
        help="Enable gradient checkpointing.",
    )

    args = p.parse_args()

    data_path = Path(args.data_jsonl)
    train_ids_path = Path(args.train_ids)
    if not data_path.exists():
        raise SystemExit(f"Data file not found: {data_path}")
    if not train_ids_path.exists():
        raise SystemExit(f"Train IDs file not found: {train_ids_path}")
    eval_ids_path = Path(args.eval_ids) if args.eval_ids.strip() else None
    if eval_ids_path and not eval_ids_path.exists():
        raise SystemExit(f"Eval IDs file not found: {eval_ids_path}")

    out_dir = Path(args.output_root) / args.run_name
    adapter_dir = out_dir / "adapter"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Lazy imports so this file can be inspected without deps installed.
    from datasets import Dataset  # type: ignore[import-untyped]
    from peft import (  # type: ignore[import-untyped]
        LoraConfig,
        TaskType,
        get_peft_model,
    )
    from transformers import (  # type: ignore[import-untyped]
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
    )
    from trl import SFTTrainer  # type: ignore[import-untyped]

    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype="auto",
        device_map="auto",
    )

    if args.gradient_checkpointing:
        model.gradient_checkpointing_enable()

    if args.target_modules.strip():
        target_modules = [t.strip() for t in args.target_modules.split(",") if t.strip()]
    else:
        target_modules = _infer_target_modules(model)

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        target_modules=target_modules,
        bias="none",
    )
    model = get_peft_model(model, lora_cfg)

    data_rows = _load_jsonl(data_path)
    data_by_id = _data_by_id(data_rows)
    train_ids = _load_id_list(train_ids_path)
    train_rows = _rows_for_ids(data_by_id, train_ids)
    if not train_rows:
        raise SystemExit("No training rows found for the given train IDs.")
    train_texts = _extract_texts(train_rows, tokenizer)
    train_ds = Dataset.from_dict({"text": train_texts})

    eval_ds = None
    if eval_ids_path:
        eval_ids = _load_id_list(eval_ids_path)
        eval_rows = _rows_for_ids(data_by_id, eval_ids)
        if eval_rows:
            eval_texts = _extract_texts(eval_rows, tokenizer)
            eval_ds = Dataset.from_dict({"text": eval_texts})

    # TrainingArguments: keep eval optional; metrics come later from held-out generation.
    # Transformers has used both `evaluation_strategy` and `eval_strategy` across versions.
    ta_kwargs = dict(
        output_dir=str(out_dir / "checkpoints"),
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        eval_steps=200 if eval_ds is not None else None,
        save_steps=200,
        logging_steps=25,
        save_total_limit=2,
        seed=args.seed,
        bf16=args.bf16,
        fp16=args.fp16,
        report_to=[],
        max_steps=-1,
    )
    eval_strategy_value = "steps" if eval_ds is not None else "no"
    try:
        import inspect

        ta_params = inspect.signature(TrainingArguments.__init__).parameters
        if "evaluation_strategy" in ta_params:
            ta_kwargs["evaluation_strategy"] = eval_strategy_value
        elif "eval_strategy" in ta_params:
            ta_kwargs["eval_strategy"] = eval_strategy_value
        else:
            ta_kwargs["evaluation_strategy"] = eval_strategy_value
    except Exception:
        ta_kwargs["evaluation_strategy"] = eval_strategy_value

    training_args = TrainingArguments(**ta_kwargs)

    # TRL's SFTTrainer signature has changed across versions. Build kwargs dynamically.
    import inspect

    sft_kwargs = dict(
        model=model,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        dataset_text_field="text",
        max_seq_length=args.max_seq_len,
        args=training_args,
        packing=False,
        # Gemma 3 (and some other models) require token_type_ids when training.
        data_collator=_make_sft_collator_with_token_type_ids(
            tokenizer, args.max_seq_len
        ),
    )
    try:
        sft_params = inspect.signature(SFTTrainer.__init__).parameters
        if "tokenizer" in sft_params:
            sft_kwargs["tokenizer"] = tokenizer
        elif "processing_class" in sft_params:
            # Newer TRL uses `processing_class` (a tokenizer/processor-like object).
            sft_kwargs["processing_class"] = tokenizer

        # Drop keys not supported by this TRL version.
        sft_kwargs = {k: v for k, v in sft_kwargs.items() if k in sft_params}
    except Exception:
        # Best-effort fallback for older versions.
        sft_kwargs["tokenizer"] = tokenizer

    trainer = SFTTrainer(**sft_kwargs)

    trainer.train()

    adapter_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))

    # Save a small run manifest for reproducibility
    manifest = {
        "base_model": args.base_model,
        "run_name": args.run_name,
        "data_jsonl": str(data_path),
        "train_ids": str(train_ids_path),
        "eval_ids": str(eval_ids_path) if eval_ids_path else "",
        "lora": {
            "r": args.lora_r,
            "alpha": args.lora_alpha,
            "dropout": args.lora_dropout,
            "target_modules": target_modules,
        },
        "training": {
            "epochs": args.epochs,
            "lr": args.lr,
            "batch_size": args.batch_size,
            "grad_accum": args.grad_accum,
            "max_seq_len": args.max_seq_len,
            "seed": args.seed,
            "bf16": args.bf16,
            "fp16": args.fp16,
            "gradient_checkpointing": args.gradient_checkpointing,
        },
    }
    with (out_dir / "run_manifest.json").open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(f"Saved adapter to: {adapter_dir}")


# Example: run from notebook (set TOKENIZERS_PARALLELISM and sys.argv then call main())
# os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# import sys
# sys.argv = ['finetune_lora.py', '--data-jsonl', 'data/title_sft.jsonl', '--train-ids', 'data/train_ids.txt', '--eval-ids', 'data/test_ids.txt', '--base-model', 'google/gemma-2-2b', '--run-name', 'title_sft_run', '--bf16']
# main()

## 3. Generate titles locally (`generate_titles_local.py`)

Generate YouTube title predictions using base model + optional LoRA adapter. Outputs JSONL for evaluation.

In [ ]:
#!/usr/bin/env python3
"""
Generate YouTube title predictions locally using a HuggingFace base model and
an optional LoRA adapter (PEFT). Pulls records from the same data JSONL used
for training, filtered by an ID list (e.g. test IDs).

Usage:
  --data-jsonl <path>   Single JSONL with all records: id, source, source_text, title.
  --input-ids <path>    One id per line for which to generate (e.g. test IDs).

Output JSONL (for evaluation):
  {"id": "...", "source": "...", "reference": "...", "prediction": "..."}

Prompts are built only in memory from the data file.
"""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path
from typing import Any, Dict, List


SYSTEM_PROMPT = """You are an expert YouTube title generator. Your task is to create compelling, accurate titles that capture the essence of video content.

Guidelines for generating YouTube titles:
1. Be accurate and truthful - the title must reflect the actual content
2. Make it compelling and click-worthy while staying honest
3. Keep titles concise (ideally 50-80 characters, max 100 characters)
4. Use natural language that viewers would search for
5. Highlight the most interesting or valuable aspect
6. Create a curiosity gap without being misleading

Generate only the title itself, nothing else."""


def _load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def _load_id_list(path: Path) -> List[str]:
    """One id per line, strip whitespace, skip empty."""
    ids: List[str] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            sid = line.strip()
            if sid:
                ids.append(sid)
    return ids


def _data_by_id(rows: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {str(r.get("id", "")): r for r in rows}


def _rows_for_ids(data_by_id: Dict[str, Dict[str, Any]], ids: List[str]) -> List[Dict[str, Any]]:
    """Return rows for given ids in order; skip missing."""
    return [data_by_id[i] for i in ids if i in data_by_id]


def _build_messages(source: str, source_text: str, reference_title: str) -> Dict[str, Any]:
    """
    Construct chat-style messages for inference, using the same pattern as training:
      - system: title-generation instructions
      - user:  instruction + source text
      - assistant (reference): the ground-truth title (used only as reference)
    """
    label = "Transcript" if source == "transcript" else "Description"
    user = (
        f"Based on the following video {label.lower()}, generate an accurate and compelling YouTube title.\n\n"
        f"{label}:\n{source_text}\n\n"
        "Generate the YouTube title:"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    reference = (reference_title or "").strip()
    if reference:
        messages.append({"role": "assistant", "content": reference})
    return {"messages": messages, "reference": reference}


def _messages_to_prompt(tokenizer: Any, messages: List[Dict[str, str]]) -> str:
    """
    Convert messages into a generation prompt.
    We *exclude* the assistant reference from the prompt so the model generates it.
    """
    # Drop any assistant messages (references) before prompting.
    prompt_messages = [m for m in messages if (m.get("role") or "").lower() != "assistant"]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    parts = []
    for m in prompt_messages:
        role = (m.get("role") or "user").upper()
        content = m.get("content") or ""
        parts.append(f"{role}:\n{content}\n")
    parts.append("ASSISTANT:\n")
    return "\n".join(parts)


def main() -> None:
    p = argparse.ArgumentParser(
        description=(
            "Generate titles locally (base model + optional LoRA adapter) using "
            "compact training records and runtime prompt construction."
        )
    )
    p.add_argument("--base-model", required=True, help="HF model name or local path.")
    p.add_argument(
        "--adapter-dir",
        default="",
        help="Optional PEFT adapter directory (from new finetune_lora.py).",
    )
    p.add_argument(
        "--data-jsonl",
        required=True,
        help="Data JSONL with all records (same as used for training).",
    )
    p.add_argument(
        "--input-ids",
        required=True,
        help="ID list to generate for, one id per line (e.g. test IDs).",
    )
    p.add_argument("--out", required=True, help="Output predictions JSONL.")
    p.add_argument("--max-new-tokens", type=int, default=32)
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--top-p", type=float, default=0.95)
    p.add_argument("--do-sample", action="store_true", help="Enable sampling (else greedy).")
    p.add_argument("--seed", type=int, default=1337)
    args = p.parse_args()

    data_path = Path(args.data_jsonl)
    input_ids_path = Path(args.input_ids)
    if not data_path.exists():
        raise SystemExit(f"Data file not found: {data_path}")
    if not input_ids_path.exists():
        raise SystemExit(f"Input IDs file not found: {input_ids_path}")

    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Lazy imports
    import torch  # type: ignore[import-untyped]
    from transformers import AutoModelForCausalLM, AutoTokenizer  # type: ignore[import-untyped]

    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype="auto",
        device_map="auto",
    )

    if args.adapter_dir.strip():
        from peft import PeftModel  # type: ignore[import-untyped]

        model = PeftModel.from_pretrained(model, args.adapter_dir.strip())

    model.eval()

    if args.do_sample:
        torch.manual_seed(args.seed)

    data_rows = _load_jsonl(data_path)
    data_by_id = _data_by_id(data_rows)
    input_ids = _load_id_list(input_ids_path)
    rows = _rows_for_ids(data_by_id, input_ids)
    if not rows:
        raise SystemExit("No rows found for the given input IDs.")

    with out_path.open("w", encoding="utf-8") as f:
        for r in rows:
            rec_id = str(r.get("id", ""))
            source = str(r.get("source", "")).strip().lower()
            source_text = r.get("source_text") or ""
            reference_title = r.get("title") or ""

            built = _build_messages(source, source_text, reference_title)
            messages = built["messages"]
            reference = built["reference"]

            prompt_text = _messages_to_prompt(tokenizer, messages)

            inputs = tokenizer(
                prompt_text,
                return_tensors="pt",
                truncation=True,
                max_length=tokenizer.model_max_length
                if tokenizer.model_max_length and tokenizer.model_max_length > 0
                else 2048,
            )
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=args.max_new_tokens,
                    do_sample=args.do_sample,
                    temperature=args.temperature if args.do_sample else None,
                    top_p=args.top_p if args.do_sample else None,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            gen = tokenizer.decode(
                out[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True
            ).strip()
            # Heuristic cleanup for titles: first line only, strip quotes.
            pred = gen.splitlines()[0].strip().strip('"').strip("'")

            f.write(
                json.dumps(
                    {
                        "id": rec_id,
                        "source": source,
                        "reference": reference,
                        "prediction": pred,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

    print(f"Wrote predictions: {out_path}")


# Example: run from notebook
# os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# import sys
# sys.argv = ['generate_titles_local.py', '--base-model', 'google/gemma-2-2b', '--adapter-dir', 'runs/title_sft_run/adapter', '--data-jsonl', 'data/title_sft.jsonl', '--input-ids', 'data/test_ids.txt', '--out', 'predictions.jsonl', '--do-sample']
# main()